In [2]:
"""
MODELO 9/12 — SAE (Stacked AutoEncoder) — VERSÃO OTIMIZADA

Replica _IM_SAE.R com otimização de performance:
- Autoencoders pré-construídos UMA VEZ por ticker
- A cada rodada WFA, apenas reseta pesos e retreina

Uso:
    python 04_model_SAE.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICSMI")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"

TRAIN_SIZE = 250
TEST_SIZE = 5

HIDDEN_LAYERS = [20, 10, 5]   # SAE: c(20,10,5)
LEARNING_RATE = 0.01
PRETRAIN_EPOCHS = 5
FINETUNE_EPOCHS = 5
BATCH_SIZE = 200
VISIBLE_DROPOUT = 0.2
HIDDEN_DROPOUT = 0.2
# ========================================================

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, InputLayer, Input
from tensorflow.keras.optimizers import Adam

tf.get_logger().setLevel("ERROR")
tf.random.set_seed(42)
np.random.seed(42)


class SAEModel:
    """
    Encapsula a rede SAE e os autoencoders de pré-treino.
    Tudo construído UMA VEZ. Apenas pesos são resetados a cada rodada.
    """

    def __init__(self, input_dim: int):
        self.input_dim = input_dim
        self.model = self._build_main_model()
        self.autoencoders = []
        self.encoder_funcs = []
        self._build_autoencoders()
        self.dense_layers = [l for l in self.model.layers
                             if isinstance(l, Dense) and l.units != 1]

    def _build_main_model(self) -> Sequential:
        model = Sequential()
        model.add(InputLayer(shape=(self.input_dim,)))
        if VISIBLE_DROPOUT > 0:
            model.add(Dropout(VISIBLE_DROPOUT))
        for h in HIDDEN_LAYERS:
            model.add(Dense(h, activation="sigmoid"))
            if HIDDEN_DROPOUT > 0:
                model.add(Dropout(HIDDEN_DROPOUT))
        model.add(Dense(1, activation="sigmoid"))
        model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                      loss="binary_crossentropy")
        return model

    def _build_autoencoders(self):
        dims = [self.input_dim] + HIDDEN_LAYERS
        for i in range(len(HIDDEN_LAYERS)):
            n_in = dims[i]
            n_out = dims[i + 1]
            ae_input = Input(shape=(n_in,))
            encoded = Dense(n_out, activation="sigmoid", name=f"ae_enc_{i}")(ae_input)
            decoded = Dense(n_in, activation="sigmoid", name=f"ae_dec_{i}")(encoded)
            ae = Model(ae_input, decoded)
            ae.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="mse")
            enc_func = Model(ae_input, encoded)
            self.autoencoders.append(ae)
            self.encoder_funcs.append(enc_func)

    def _reset_layer_weights(self, layer):
        if hasattr(layer, "kernel_initializer") and hasattr(layer, "kernel"):
            layer.kernel.assign(layer.kernel_initializer(layer.kernel.shape))
        if hasattr(layer, "bias_initializer") and hasattr(layer, "bias"):
            layer.bias.assign(layer.bias_initializer(layer.bias.shape))

    def reset_and_pretrain(self, X_train: np.ndarray):
        for layer in self.model.layers:
            self._reset_layer_weights(layer)
        self.model.optimizer = Adam(learning_rate=LEARNING_RATE)

        current_input = X_train.copy()
        batch = min(BATCH_SIZE, len(X_train))

        for ae, enc_func, dense_layer in zip(
                self.autoencoders, self.encoder_funcs, self.dense_layers):
            for layer in ae.layers:
                self._reset_layer_weights(layer)
            ae.optimizer = Adam(learning_rate=LEARNING_RATE)

            ae.fit(current_input, current_input,
                   epochs=PRETRAIN_EPOCHS, batch_size=batch, verbose=0)

            encoder_layer = ae.layers[1]
            dense_layer.set_weights(encoder_layer.get_weights())

            current_input = enc_func.predict(current_input, verbose=0)

    def finetune(self, X_train: np.ndarray, y_train: np.ndarray):
        self.model.fit(X_train, y_train,
                       epochs=FINETUNE_EPOCHS,
                       batch_size=min(BATCH_SIZE, len(X_train)),
                       verbose=0)

    def predict(self, X_test: np.ndarray) -> np.ndarray:
        probs = self.model.predict(X_test, verbose=0).reshape(-1)
        return (probs >= 0.5).astype(int)


def read_codes(path: Path) -> list:
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    return df["Code"].str.strip().str.upper().tolist()


def run_wfa_sae(code: str, base_dir: Path) -> dict:
    infile = base_dir / f"{code}_DatasetNew_MI.csv"
    outfile = base_dir / f"{code}_TradeSignal_SAE_MI.csv"

    if outfile.exists():
        return {"Code": code, "status": "skipped", "signals": 0}
    if not infile.exists():
        return {"Code": code, "status": "no_DatasetNew", "signals": 0}

    try:
        df = pd.read_csv(infile, encoding="utf-8-sig")
    except Exception as e:
        return {"Code": code, "status": f"read_error: {e}", "signals": 0}

    if df.shape[1] < 3:
        return {"Code": code, "status": "too_few_columns", "signals": 0}

    date_col = df.columns[0]
    label_col = df.columns[-1]
    feature_cols = df.columns[1:-1].tolist()
    input_dim = len(feature_cols)

    M = len(df)
    if M < TRAIN_SIZE + TEST_SIZE:
        return {"Code": code, "status": f"too_few_rows ({M})", "signals": 0}

    Q = (M - TRAIN_SIZE) // TEST_SIZE
    H = (M - TRAIN_SIZE) - TEST_SIZE * Q
    df = df.iloc[H:].reset_index(drop=True)

    dates = df[date_col].values
    X = df[feature_cols].values.astype(float)
    y = df[label_col].values.astype(int)

    sae = SAEModel(input_dim)

    predict_signal = []
    predict_dates = []

    for i in range(Q):
        train_start = i * TEST_SIZE
        train_end = train_start + TRAIN_SIZE
        test_start = train_end
        test_end = test_start + TEST_SIZE

        X_train = X[train_start:train_end]
        y_train = y[train_start:train_end]
        X_test = X[test_start:test_end]
        test_dates_i = dates[test_start:test_end]

        if len(np.unique(y_train)) < 2:
            preds = [int(y_train[0])] * len(X_test)
        else:
            try:
                sae.reset_and_pretrain(X_train)
                sae.finetune(X_train, y_train)
                preds = sae.predict(X_test).tolist()
            except Exception:
                preds = [0] * len(X_test)

        predict_signal.extend(preds)
        predict_dates.extend(test_dates_i)

    tf.keras.backend.clear_session()

    if predict_signal:
        df_out = pd.DataFrame({"Date": predict_dates, "pre_signal": predict_signal})
        df_out.to_csv(outfile, index=False, encoding="utf-8-sig")

    return {"Code": code, "status": "ok", "signals": len(predict_signal)}


def main():
    codes = read_codes(SEC_NAMES)
    print(f"Modelo: SAE (hidden={HIDDEN_LAYERS}, sigmoid, lr={LEARNING_RATE})")
    print(f"WFA: d1={TRAIN_SIZE}, d2={TEST_SIZE}")
    print(f"Pretrain={PRETRAIN_EPOCHS}, Finetune={FINETUNE_EPOCHS}, batch={BATCH_SIZE}")
    print(f"Otimização: autoencoders pré-construídos (reset apenas pesos)")
    print(f"Tickers: {len(codes)}\n")

    report = []
    for code in tqdm(codes, desc="SAE Walk-Forward"):
        result = run_wfa_sae(code, BASE_DIR)
        report.append(result)

    report_df = pd.DataFrame(report)
    report_df.to_csv(BASE_DIR / "model_SAE_reportMI.csv", index=False, encoding="utf-8-sig")

    n_ok = (report_df["status"] == "ok").sum()
    n_skip = (report_df["status"] == "skipped").sum()
    avg_signals = report_df.loc[report_df["status"] == "ok", "signals"].mean()

    print(f"\n{'='*50}")
    print(f"Concluído: {n_ok} processados, {n_skip} já existiam.")
    print(f"Média de sinais por ação: {avg_signals:.0f}")
    print(f"Relatório: model_SAE_report.csv")


if __name__ == "__main__":
    main()

Modelo: SAE (hidden=[20, 10, 5], sigmoid, lr=0.01)
WFA: d1=250, d2=5
Pretrain=5, Finetune=5, batch=200
Otimização: autoencoders pré-construídos (reset apenas pesos)
Tickers: 239



SAE Walk-Forward: 100%|██████████| 239/239 [10:34:19<00:00, 159.24s/it] 


Concluído: 239 processados, 0 já existiam.
Média de sinais por ação: 564
Relatório: model_SAE_report.csv
